# LoRA fine-tune: Qwen3-4B rationale explainer

**Inputs:** `train.jsonl` generated locally by `src/copilot/reasoning/datagen.py`.

**Output:** a LoRA adapter you download and drop into `artifacts/lora_adapter/`.

In [4]:
!pip -q install -U "transformers>=4.51" "peft>=0.12" "trl>=0.11" "datasets>=2.20" "accelerate>=0.34" bitsandbytes
import torch
print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

CUDA available: True | NVIDIA L4


In [5]:
from google.colab import files
uploaded = files.upload()   # train.jsonl
DATA_PATH = next(iter(uploaded))
print('uploaded:', DATA_PATH)

Saving train.jsonl to train.jsonl
uploaded: train.jsonl


In [6]:
from datasets import load_dataset
ds = load_dataset('json', data_files=DATA_PATH, split='train')
print(ds)
print(ds[0]['messages'][2]['content'][:300])

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['messages'],
    num_rows: 321
})
Why Empagliflozin?
- Guideline-recommended first-line therapy for Type 2 Diabetes (ADA Standards of Care in Diabetes).
- No similar-patient outcome data was available for this option.
- No contraindications, drug interactions, or allergy conflicts were detected for this patient.
- Compatible with co


In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen3-4B-Instruct-2507'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
model.config.use_cache = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Each row already has `system` / `user` / `assistant` turns. We render them to a single string the trainer learns to complete.

In [8]:
def to_text(example):
    return {'text': tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False)}

ds_text = ds.map(to_text, remove_columns=ds.column_names)
print(ds_text[0]['text'][:600])

Map:   0%|          | 0/321 [00:00<?, ? examples/s]

<|im_start|>system
You are a clinical decision-support explainer. A deterministic ranking system has ALREADY selected the recommended treatment. Your ONLY job is to explain why the selected drug was recommended, using ONLY the structured facts provided. You must NOT change the recommendation, suggest a different drug, invent facts, or add clinical claims that are not in the facts. Write 4 concise bullet points in this exact shape:
Why <DRUG>?
- Guideline basis (cite the source).
- Similar-patient evidence.
- Safety check result.
- Comorbidity and lab compatibility.
Keep it factual, neutral, an


In [9]:
# lora configs
from peft import LoraConfig

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)

In [12]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [13]:
# train SFTTrainer
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir='qwen3-4b-rationale-lora',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    logging_steps=10,
    bf16=True,
    max_length=1024,
    dataset_text_field='text',
    packing=False,
    report_to='none',
)

trainer = SFTTrainer(
    model=model, args=args, train_dataset=ds_text, peft_config=peft_config,
    processing_class=tokenizer,
)
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/321 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/321 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/321 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.774670
20,0.311345
30,0.092231
40,0.084359
50,0.072097
60,0.071149
70,0.069226
80,0.068763
90,0.068494
100,0.062923


TrainOutput(global_step=123, training_loss=0.22952354438905795, metrics={'train_runtime': 749.5041, 'train_samples_per_second': 1.285, 'train_steps_per_second': 0.164, 'total_flos': 1.26524300788224e+16, 'train_loss': 0.22952354438905795, 'entropy': 0.06399364893635114, 'num_tokens': 547404.0, 'mean_token_accuracy': 0.9756419989797804, 'epoch': 3.0})

In [14]:
ADAPTER_DIR = 'lora_adapter'
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('saved adapter to', ADAPTER_DIR)

saved adapter to lora_adapter


In [15]:
# sanity check (explain-only behavior)
from peft import PeftModel
eval_model = PeftModel.from_pretrained(model, ADAPTER_DIR)
eval_model.eval()

test_msgs = ds[0]['messages'][:2]   # system + user, no assistant
prompt = tokenizer.apply_chat_template(test_msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(eval_model.device)
out = eval_model.generate(**inputs, max_new_tokens=180, do_sample=False)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Why Empagliflozin?
- Guideline-recommended first-line therapy for Type 2 Diabetes (ADA Standards of Care in Diabetes).
- No similar-patient outcome data was available for this option.
- No contraindications, drug interactions, or allergy conflicts were detected for this patient.
- Compatible with comorbid the diabetes profile and the patient's current labs.


In [16]:
!zip -r lora_adapter.zip lora_adapter > /dev/null
from google.colab import files
files.download('lora_adapter.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Locally:** unzip into `artifacts/lora_adapter/` so the structure is `artifacts/lora_adapter/adapter_config.json` etc. Then the pipeline's `LoRARationaleGenerator` will load it automatically.